# Environment Setup

This notebook verifies your OpenShift environment, deploys models on RHOAI, and tests InferenceService endpoints.

## 1. Verify Cluster Access

In [8]:
import subprocess
import json

print("=" * 50)
print("OpenShift Environment Verification")
print("=" * 50)

checks = [
    ("oc CLI", ["oc", "version", "--client", "-o", "json"]),
    ("Cluster login", ["oc", "whoami"]),
    ("Cluster URL", ["oc", "whoami", "--show-server"]),
]

for name, cmd in checks:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        output = result.stdout.strip()
        if result.returncode == 0:
            print(f"✅ {name}: {output[:80]}")
        else:
            print(f"❌ {name}: {result.stderr.strip()[:80]}")
    except FileNotFoundError:
        print(f"❌ {name}: command not found")

OpenShift Environment Verification
✅ oc CLI: {
  "clientVersion": {
    "major": "",
    "minor": "",
    "gitVersion": "4.21
✅ Cluster login: kube:admin
✅ Cluster URL: https://api.openshift-cluster.sandbox1785.opentlc.com:6443


## 2. Verify RHOAI Operator

In [9]:
%%bash
echo "=== RHOAI Operator ==="
oc get csv -n redhat-ods-operator 2>/dev/null | grep -i rhods || echo "⚠️  RHOAI operator not found"

echo ""
echo "=== GPU Nodes ==="
oc get nodes -l nvidia.com/gpu.present=true --no-headers 2>/dev/null || echo "⚠️  No GPU nodes labeled"

echo ""
echo "=== KServe/ModelMesh ==="
oc get crd inferenceservices.serving.kserve.io --no-headers 2>/dev/null && echo "✅ KServe CRD available" || echo "❌ KServe not installed"

=== RHOAI Operator ===
rhods-operator.3.4.0            Red Hat OpenShift AI                          3.4.0     rhods-operator.3.3.2            Succeeded

=== GPU Nodes ===
ip-10-0-99-191.us-east-2.compute.internal   Ready   gpu-worker,worker   18d   v1.33.11

=== KServe/ModelMesh ===
inferenceservices.serving.kserve.io   2026-05-23T13:26:42Z
✅ KServe CRD available


## 3. Configure Environment Variables

In [10]:
from pathlib import Path

env_path = Path("../.env")
sample_path = Path("../sample.env")

if not env_path.exists():
    if sample_path.exists():
        env_path.write_text(sample_path.read_text())
        print(f"Created {env_path} from sample.env")
        print("⚠️  Edit .env and fill in your tokens before proceeding.")
    else:
        print("❌ sample.env not found.")
else:
    print(f"✅ {env_path} already exists.")

✅ ../.env already exists.


## 4. Deploy or Select a Model

This lab requires an OpenAI-compatible model endpoint. Two options:

**Option A (Default)** — Deploy a Qwen model via **LLMInferenceService (llm-d)** with MaaS Gateway:

| Model | VRAM | Deployment | MaaS | Source | Notes |
|-------|------|-----------|:----:|--------|-------|
| `qwen3-14b` | ~14 GB | llm-d | Yes | OCI modelcar | **Default.** Dense, FP8, reasoning + tool-calling |
| `qwen-coder-7b` | ~8 GB | llm-d | Yes | OCI/HF | Code-focused (Qwen2.5-Coder) |
| `qwen-coder-14b` | ~16 GB | llm-d | Yes | OCI/HF | Better code quality (Qwen2.5-Coder) |
| `qwen3-coder-30b` | ~24 GB | llm-d | Verify | HF | Qwen3 MoE — verify RHOAI 3.4 MoE support |
| `qwen36-35b-a3b` | ~21 GB | InferenceService | No | OCI custom | Qwen3.6 MoE — no MaaS integration |

> **Deployment types:**
> - **llm-d** = `LLMInferenceService` — native MaaS integration, automatic inference-gateway registration, API key management, rate limiting
> - **InferenceService** = Standard KServe — direct route only, no MaaS features
>
> **OCI modelcar** = Model weights baked into a container image (fast startup, no download at runtime)

**Option B** — Use an existing model already deployed in any namespace (e.g. `gemma4` in another project). Set `USE_EXISTING = True` below and fill in the endpoint info.

In [11]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📌 Configuration — edit this section to match your environment
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Option A: Deploy a new Qwen model via LLMInferenceService (llm-d)
DEPLOY_MODEL = "qwen36-27b"  # "qwen36-27b" | "qwen-coder-7b" | "qwen-coder-14b" | "qwen3-coder-30b" | "qwen36-35b-a3b"

# Option B: Use an existing model already deployed on the cluster
USE_EXISTING = False  # Set True to skip deployment and use existing model
EXISTING_NAMESPACE = "my-models"        # namespace where existing model lives
EXISTING_ISVC_NAME = "gemma4"           # InferenceService or LLMInferenceService name
EXISTING_MODEL_NAME = "gemma-3-4b-it"   # model name for API calls (served-model-name)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def get_llmisvc_info(name, namespace):
    """Get URL and readiness of an LLMInferenceService (llm-d)."""
    r = subprocess.run(
        ["oc", "get", "llminferenceservice", name, "-n", namespace, "-o", "json"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        return None, None
    data = json.loads(r.stdout)
    url = data.get("status", {}).get("url", "")
    conditions = data.get("status", {}).get("conditions", [])
    ready = next((c["status"] for c in conditions if c["type"] == "Ready"), "Unknown")
    return url, ready

def get_isvc_info(name, namespace):
    """Get URL and readiness of an InferenceService (fallback)."""
    r = subprocess.run(
        ["oc", "get", "inferenceservice", name, "-n", namespace, "-o", "json"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        return None, None
    data = json.loads(r.stdout)
    url = data.get("status", {}).get("url", "")
    conditions = data.get("status", {}).get("conditions", [])
    ready = next((c["status"] for c in conditions if c["type"] == "Ready"), "Unknown")
    return url, ready

def get_deployed_models(namespace):
    """Return list of LLMInferenceService + InferenceService names."""
    models = []
    for kind in ["llminferenceservice", "inferenceservice"]:
        r = subprocess.run(
            ["oc", "get", kind, "-n", namespace, "-o", "json"],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            for item in json.loads(r.stdout).get("items", []):
                models.append((item["metadata"]["name"], kind))
    return models

# --- Main logic ---

if USE_EXISTING:
    url, ready = get_llmisvc_info(EXISTING_ISVC_NAME, EXISTING_NAMESPACE)
    if not url:
        url, ready = get_isvc_info(EXISTING_ISVC_NAME, EXISTING_NAMESPACE)
    if url:
        MODEL_NAME = EXISTING_MODEL_NAME
        MODEL_URL = url
        MODEL_NAMESPACE = EXISTING_NAMESPACE
        print(f"✅ Using existing model: {EXISTING_ISVC_NAME} in namespace '{EXISTING_NAMESPACE}'")
        print(f"   URL:   {url}")
        print(f"   Model: {MODEL_NAME}")
        print(f"   Ready: {ready}")
    else:
        print(f"❌ Model '{EXISTING_ISVC_NAME}' not found in namespace '{EXISTING_NAMESPACE}'.")
        print(f"   Check with: oc get llminferenceservice,inferenceservice -n {EXISTING_NAMESPACE}")
        MODEL_NAME = MODEL_URL = MODEL_NAMESPACE = None
else:
    NAMESPACE = "demo"
    MODEL_MANIFEST_MAP = {
        "qwen3-14b":       "manifests/07-model-qwen3-14b-fp8.yaml",
        "qwen-coder-7b":   "manifests/01-model-qwen-coder-7b.yaml",
        "qwen-coder-14b":  "manifests/02-model-qwen-coder-14b.yaml",
        "qwen3-coder-30b": "manifests/03-model-qwen3-coder-30b.yaml",
        "qwen36-35b-a3b":  "manifests/06-model-qwen36-35b-a3b-oci.yaml",
    }
    deployed = get_deployed_models(NAMESPACE)

    if deployed:
        print(f"✅ Models already deployed in '{NAMESPACE}':")
        for name, kind in deployed:
            if kind == "llminferenceservice":
                url, ready = get_llmisvc_info(name, NAMESPACE)
                print(f"   • {name}  [llm-d] (Ready={ready})")
            else:
                url, ready = get_isvc_info(name, NAMESPACE)
                print(f"   • {name}  [InferenceService] (Ready={ready})")
        print("\n⏭️  Skipping deployment.")
        print("   To redeploy: oc delete llminferenceservice,inferenceservice --all -n demo")
    else:
        print(f"No models in '{NAMESPACE}'. Deploying {DEPLOY_MODEL} via llm-d...")
        print("")
        subprocess.run(["oc", "apply", "-f", "manifests/00-demo-ns.yaml"],
                       capture_output=True, text=True)
        hf_token = os.getenv("HF_TOKEN", "")
        if hf_token and not hf_token.startswith("REPLACE"):
            pipe = subprocess.run(
                ["oc", "create", "secret", "generic", "hf-token",
                 f"--from-literal=token={hf_token}", "-n", NAMESPACE,
                 "--dry-run=client", "-o", "yaml"],
                capture_output=True, text=True)
            subprocess.run(["oc", "apply", "-f", "-"], input=pipe.stdout,
                           capture_output=True, text=True)
        model_manifest = MODEL_MANIFEST_MAP[DEPLOY_MODEL]
        r = subprocess.run(["oc", "apply", "-f", model_manifest],
                           capture_output=True, text=True)
        print(r.stdout)
        print(f"✅ Deployed {DEPLOY_MODEL} (LLMInferenceService)")
        print("⏳ Pulling OCI modelcar image (3-8 min depending on network).")
        print("   Monitor: oc get pods -n demo -w")

    url, ready = get_llmisvc_info(DEPLOY_MODEL, NAMESPACE)
    if not url:
        url, ready = get_isvc_info(DEPLOY_MODEL, NAMESPACE)
    MODEL_NAME = DEPLOY_MODEL
    MODEL_URL = url or f"(pending — wait for {DEPLOY_MODEL} to become Ready)"
    MODEL_NAMESPACE = NAMESPACE

print("")
print("=" * 60)
print(f"  LAB MODEL CONFIG (used in subsequent notebooks):")
print(f"    MODEL_NAME:      {MODEL_NAME}")
print(f"    MODEL_URL:       {MODEL_URL}")
print(f"    MODEL_NAMESPACE: {MODEL_NAMESPACE}")
print("=" * 60)

✅ Models already deployed in 'demo':
   • gemma4-12b  [llm-d] (Ready=False)
   • qwen3-4b  [llm-d] (Ready=False)
   • qwen36-27b  [llm-d] (Ready=True)

⏭️  Skipping deployment.
   To redeploy: oc delete llminferenceservice,inferenceservice --all -n demo

  LAB MODEL CONFIG (used in subsequent notebooks):
    MODEL_NAME:      qwen36-27b
    MODEL_URL:       https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b
    MODEL_NAMESPACE: demo


### Save discovered values to `.env`

The cell below persists the model configuration discovered above back to `../.env`.
Subsequent notebooks (`2_app_setup`, `3_maas/`, etc.) will load these values automatically via `dotenv`.

In [12]:
import subprocess, os, re

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env") if os.path.basename(os.getcwd()) == "0_setup" else "../.env"
env_path = os.path.abspath(env_path)

# Auto-detect CLUSTER_DOMAIN from the cluster
cluster_domain_r = subprocess.run(
    ["oc", "get", "ingresses.config.openshift.io", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = cluster_domain_r.stdout.strip() if cluster_domain_r.returncode == 0 else ""

# Values to persist
updates = {
    "CLUSTER_DOMAIN": CLUSTER_DOMAIN,
    "MODEL_NAME": MODEL_NAME or "",
    "MODEL_NAMESPACE": MODEL_NAMESPACE or "",
    "MODEL_ENDPOINT": MODEL_URL or "",
}

# Read existing .env or start fresh
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        content = f.read()
else:
    content = ""

changed = []
for key, value in updates.items():
    if not value or value.startswith("(pending"):
        continue
    pattern = rf"^#?\s*{key}=.*$"
    replacement = f"{key}={value}"
    if re.search(pattern, content, re.MULTILINE):
        new_content = re.sub(pattern, replacement, content, count=1, flags=re.MULTILINE)
        if new_content != content:
            content = new_content
            changed.append(key)
    else:
        content = content.rstrip("\n") + f"\n{replacement}\n"
        changed.append(key)

if changed:
    with open(env_path, "w") as f:
        f.write(content)
    print(f"✅ Updated {env_path}:")
    for k in changed:
        print(f"   {k}={updates[k]}")
else:
    print(f"✅ .env already up-to-date (no changes needed).")

print(f"\n📋 Current model config in .env:")
for key in ["CLUSTER_DOMAIN", "MODEL_NAME", "MODEL_NAMESPACE", "MODEL_ENDPOINT"]:
    print(f"   {key}={updates.get(key, '(not set)')}")

✅ .env already up-to-date (no changes needed).

📋 Current model config in .env:
   CLUSTER_DOMAIN=apps.openshift-cluster.sandbox1785.opentlc.com
   MODEL_NAME=qwen36-27b
   MODEL_NAMESPACE=demo
   MODEL_ENDPOINT=https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b


In [5]:
import subprocess

ns = MODEL_NAMESPACE or "demo"

print(f"=== LLMInferenceService (llm-d) in '{ns}' ===")
r = subprocess.run(["oc", "get", "llminferenceservice", "-n", ns], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 and r.stdout.strip() else "(none)")

print(f"\n=== InferenceService in '{ns}' ===")
r = subprocess.run(["oc", "get", "inferenceservice", "-n", ns], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 and r.stdout.strip() else "(none)")

print(f"\n=== Pods in '{ns}' ===")
r = subprocess.run(["oc", "get", "pods", "-n", ns], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "(no pods)")

print("\nTip: Re-run this cell until READY=True, then proceed to Step 5.")

=== LLMInferenceService (llm-d) in 'demo' ===
NAME         URL                                                                                        READY   REASON    AGE
gemma4-12b   https://inference-gateway.apps.openshift-cluster.sandbox1785.opentlc.com/demo/gemma4-12b   False   Stopped   7d19h
qwen3-4b     https://inference-gateway.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen3-4b     False   Stopped   20d
qwen36-27b   https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b            True              6d8h


=== InferenceService in 'demo' ===
(none)

=== Pods in 'demo' ===
NAME                                                 READY   STATUS                        RESTARTS        AGE
create-bucket-7pntj                                  1/1     Running                       0               5h7m
create-bucket-c2qd5                                  0/1     ContainerStatusUnknown        1               3d22h
create-bucket-fzh9h                        

## 5. Test Model Endpoints

Once InferenceServices show `READY=True`, test the endpoints.

> **Note:** We use `oc port-forward` to access the model directly via its internal TLS service,
> bypassing the MaaS gateway's `payload-processing` ext_proc filter which intercepts HTTP traffic
> on the `maas-api` hostname.

In [13]:
import subprocess, json, time

ns = MODEL_NAMESPACE or "demo"

if not MODEL_URL or MODEL_URL.startswith("(pending"):
    print(f"⚠️  Model URL not resolved yet. Wait for deployment to complete.")
    print(f"   Re-run Step 4 after the model is Ready.")
else:
    svc_name = f"{MODEL_NAME}-kserve-workload-svc"
    local_port = 18000
    print(f"Testing endpoint via port-forward → svc/{svc_name}:8000")
    print(f"Model: {MODEL_NAME}")
    print("=" * 60)

    pf = subprocess.Popen(
        ["oc", "port-forward", f"svc/{svc_name}", f"{local_port}:8000", "-n", ns],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(3)

    try:
        test_result = subprocess.run(
            ["curl", "-sk", "--max-time", "10",
             f"https://localhost:{local_port}/v1/models"],
            capture_output=True, text=True, timeout=15
        )

        if test_result.returncode == 0 and test_result.stdout.strip():
            try:
                models = json.loads(test_result.stdout)
                print(f"✅ Responding! Available models: {[m['id'] for m in models.get('data', [])]}")
            except json.JSONDecodeError:
                print(f"⚠️  Got response but not valid JSON:\n{test_result.stdout[:200]}")
        else:
            print(f"❌ Not responding yet. Check pod status:")
            print(f"   oc get pods -n {ns}")
    finally:
        pf.terminate()
        pf.wait()

Testing endpoint via port-forward → svc/qwen36-27b-kserve-workload-svc:8000
Model: qwen36-27b
✅ Responding! Available models: ['qwen36-27b']


In [14]:
import subprocess, json, time, signal, os

ns = MODEL_NAMESPACE or "demo"

if not MODEL_URL or MODEL_URL.startswith("(pending"):
    print("⚠️  Model not ready. Re-run Step 4 status check.")
else:
    print(f"Inference test → {MODEL_NAME}")
    print("-" * 50)

    svc_name = f"{MODEL_NAME}-kserve-workload-svc"
    local_port = 18000

    # Start port-forward to bypass MaaS gateway ext_proc filter
    pf = subprocess.Popen(
        ["oc", "port-forward", f"svc/{svc_name}", f"{local_port}:8000", "-n", ns],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(3)

    try:
        r = subprocess.run(
            ["curl", "-sk", "--max-time", "30",
             f"https://localhost:{local_port}/v1/chat/completions",
             "-H", "Content-Type: application/json",
             "-d", json.dumps({
                 "model": MODEL_NAME,
                 "messages": [{"role": "user", "content": "Write a Python hello world one-liner."}],
                 "max_tokens": 30
             })],
            capture_output=True, text=True, timeout=35
        )
        if r.returncode == 0 and r.stdout.strip():
            try:
                resp = json.loads(r.stdout)
                if "choices" in resp:
                    content = resp["choices"][0]["message"]["content"]
                    print(f"✅ Response:\n{content}")
                elif "error" in resp:
                    print(f"❌ API error: {resp['error']}")
                else:
                    print(f"❌ Unexpected response:\n{r.stdout[:300]}")
            except json.JSONDecodeError:
                print(f"❌ Non-JSON response (model may still be loading):\n{r.stdout[:300]}")
        else:
            print(f"❌ No response. returncode={r.returncode}")
            if r.stderr:
                print(f"   stderr: {r.stderr[:200]}")
    finally:
        pf.terminate()
        pf.wait()

Inference test → qwen36-27b
--------------------------------------------------
✅ Response:
Here's a thinking process:

1.  **Analyze User Request:**
   - **Language:** Python
   - **Task:** Hello


## 6. Verify MaaS Availability

Check if MaaS (Models as a Service) is available on your cluster.

In [16]:
%%bash
echo "=== MaaS Gateway ==="
oc get gateway -n openshift-ingress maas-default-gateway 2>/dev/null && echo "✅ MaaS Gateway found" || echo "⚠️  MaaS Gateway not found — install MaaS via RHOAI operator"

echo ""
echo "=== MaaS Endpoint ==="
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}' 2>/dev/null)
if [ -n "$CLUSTER_DOMAIN" ]; then
    echo "  https://maas-api.${CLUSTER_DOMAIN}"
else
    echo "⚠️  Could not determine cluster domain"
fi

=== MaaS Gateway ===
NAME                   CLASS                          ADDRESS                                                                   PROGRAMMED   AGE
maas-default-gateway   openshift-gateway-controller   a51c70bf9526e46bfa46531fabca8a01-2141190600.us-east-2.elb.amazonaws.com   True         25d
✅ MaaS Gateway found

=== MaaS Endpoint ===
  https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com


## 7. Install Observability Operators

The monitoring notebook (`6_monitoring/`) requires three operators.
This cell checks if they are already installed and installs any missing ones via Subscription CR.

| Operator | Package | Purpose |
|----------|---------|---------|
| **Tempo Operator** | `tempo-product` | Distributed trace storage (TempoMonolithic) |
| **Red Hat build of OpenTelemetry** | `opentelemetry-product` | OTel Collector CR management |
| **Cluster Observability Operator** | `cluster-observability-operator` | Console Observe → Traces plugin |

In [1]:
%%bash
OPERATORS=(
    "tempo-product|openshift-tempo-operator|tempo"
    "opentelemetry-product|openshift-opentelemetry-operator|opentelemetry"
    "cluster-observability-operator|openshift-operators|cluster-observability-operator"
)

for entry in "${OPERATORS[@]}"; do
    IFS='|' read -r PKG NS GREP_NAME <<< "$entry"
    CSV=$(oc get csv -n "$NS" --no-headers 2>/dev/null | grep "$GREP_NAME" | grep Succeeded | head -1)
    if [ -n "$CSV" ]; then
        echo "✅ $PKG: already installed ($(echo $CSV | awk '{print $1}'))"
    else
        echo "⏳ $PKG: not found — installing..."
        if [ "$NS" != "openshift-operators" ]; then
            oc create namespace "$NS" --dry-run=client -o yaml 2>/dev/null | oc apply -f - 2>/dev/null
            oc apply -f - <<EOF 2>/dev/null
apiVersion: operators.coreos.com/v1
kind: OperatorGroup
metadata:
  name: $NS
  namespace: $NS
spec:
  upgradeStrategy: Default
EOF
        fi
        oc apply -f - <<EOF
apiVersion: operators.coreos.com/v1alpha1
kind: Subscription
metadata:
  name: $PKG
  namespace: $NS
spec:
  channel: stable
  name: $PKG
  source: redhat-operators
  sourceNamespace: openshift-marketplace
  installPlanApproval: Automatic
EOF
    fi
done

echo ""
echo "Waiting for all operators to become ready (up to 5 min)..."
ALL_READY=false
i=0
while [ $i -lt 60 ]; do
    i=$((i+1))
    READY_COUNT=0
    for entry in "${OPERATORS[@]}"; do
        IFS='|' read -r PKG NS GREP_NAME <<< "$entry"
        if oc get csv -n "$NS" --no-headers 2>/dev/null | grep "$GREP_NAME" | grep -q Succeeded; then
            READY_COUNT=$((READY_COUNT+1))
        fi
    done
    if [ "$READY_COUNT" -eq 3 ]; then
        ALL_READY=true
        break
    fi
    sleep 5
done

echo ""
if $ALL_READY; then
    echo "✅ All observability operators installed:"
    for entry in "${OPERATORS[@]}"; do
        IFS='|' read -r PKG NS GREP_NAME <<< "$entry"
        oc get csv -n "$NS" --no-headers 2>/dev/null | grep "$GREP_NAME" | grep Succeeded | awk '{printf "   %s (%s)\n", $1, $NF}'
    done
else
    echo "⚠️  Some operators not ready yet. Check manually:"
    echo "   oc get csv -A | grep -E 'tempo|opentelemetry|observability'"
    echo "   If InstallPlan is pending approval: oc get installplan -n openshift-operators"
fi

✅ tempo-product: already installed (tempo-operator.v0.21.0-1)
✅ opentelemetry-product: already installed (opentelemetry-operator.v0.152.0-1)
✅ cluster-observability-operator: already installed (cluster-observability-operator.v1.5.0)

Waiting for all operators to become ready (up to 5 min)...

✅ All observability operators installed:
   tempo-operator.v0.21.0-1 (Succeeded)
   opentelemetry-operator.v0.152.0-1 (Succeeded)
   cluster-observability-operator.v1.5.0 (Succeeded)


## Next Steps

Once models are ready (`oc get llminferenceservice -n demo` shows Ready):

1. **Phase 1** → `1_mcp_servers/2_deploy_mcp_servers.ipynb` to deploy MCP tool servers
2. **Phase 2** → `3_maas/2_enable_maas.ipynb` to enable Models as a Service (PostgreSQL + Gateway + API keys)
3. **Monitoring** → `6_monitoring/1_observability_setup.ipynb` to deploy Grafana dashboards + distributed tracing

> **Note:** If you have an existing PostgreSQL, set `MAAS_DB_CONNECTION_URL` in `.env` before running Phase 2.